### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sberbank_housing_market_forecasting",
    dataset_year="2017",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/sberbank-russian-housing-market",
    download_description="""
We use the data from Kaggle:

kaggle competitions download -c sberbank-russian-housing-market
mkdir -p local-data-warehouse/sberbank_housing_market_forecasting && mv sberbank-russian-housing-market.zip local-data-warehouse/sberbank_housing_market_forecasting/ && cd local-data-warehouse/sberbank_housing_market_forecasting/ && unzip sberbank-russian-housing-market.zip && rm test.csv.zip sample_submission.csv.zip data_dictionary.txt sberbank-russian-housing-market.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{Herman2024HomeCreditCreditRiskModelStability,
  author = {Daniel Herman and Tomas Jelinek and Walter Reade and Maggie Demkin and Addison Howard},
  title  = {Home Credit - Credit Risk Model Stability},
  year   = {2024},
  howpublished = {\url{https://kaggle.com/competitions/home-credit-credit-risk-model-stability}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Herman2024HomeCreditCreditRiskModelStability",
    license="Kaggle Competition Rules",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
We follow the preprocessing from TabRed (https://github.com/yandex-research/tabred/tree/main/preprocessing#sberbank-housing-market-forecasting).

- We drop two duplicated columns with the same values as other columns: "0_6_all", "7_14_all"
- We drop the ID column as it does not provide information.
- Note, the dataset contains a lot of spatial feature and even more already decoded spatial information (like distances to various points of interest).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="price_doc",
    problem_type="regression",
    # We use RMSE instead of RMSLE from the competition as we already log scale the target
    objective_metric_name="rmse",
    time_on="timestamp",
)

## Preprocessing

In [2]:
import pandas as pd
import polars as pl
import zipfile
from pathlib import Path

# From: https://github.com/yandex-research/tabred/blob/main/preprocessing/sberbank-housing.py
data = pl.read_csv(
    zipfile.ZipFile(dataset_mold.path / 'train.csv.zip').read('train.csv'),
    null_values=["NA"],
    infer_schema_length=30_000
).with_columns(
    pl.col('timestamp').str.strptime(pl.Date)
)
data_macro = pl.read_csv(
    zipfile.ZipFile(dataset_mold.path / 'macro.csv.zip').read('macro.csv'),
    null_values=["NA"],
    infer_schema_length=30_000
).with_columns(
    pl.col('timestamp').str.strptime(pl.Date),
    pl.col('child_on_acc_pre_school').str.replace(',', '.').cast(pl.Float32, strict=False),
    pl.col('modern_education_share').str.replace(',', '.').cast(pl.Float32),
    pl.col('old_education_build_share').str.replace(',', '.').cast(pl.Float32),
).drop('provision_retail_space_modern_sqm') # this feature has one value except for nulls
data_fixup = (
    pl.read_excel(Path.cwd() / "BAD_ADDRESS_FIX.xlsx")
      .with_columns(pl.col(pl.Utf8).replace("NA", None))
)

data = data.filter(
    pl.col('kremlin_km').ne(pl.col('kremlin_km').min()) |
    pl.col('id').is_in(data_fixup['id'])
).update(data_fixup, on='id')
data = data.filter(
    pl.col('full_sq').gt(5.0) &
    pl.col('full_sq').ne(5326.0) &
    pl.col('price_doc').gt(1_000_000) &
    pl.col('price_doc').ne(2_000_000) &
    pl.col('price_doc').ne(3_000_000)
)

data = data.join(data_macro, on="timestamp")

# log scale target as in TabRed
data = data.with_columns(
    (pl.col("price_doc") / pl.col("full_sq")).log().alias("price_doc")
)

data = data.to_pandas()
cat_cols = [
    'ID_railroad_station_walk','ID_railroad_station_avto','ID_big_road1','ID_big_road2','ID_railroad_terminal','ID_bus_terminal','ecology','material','state','sub_area','product_type', 'culture_objects_top_25', 'thermal_power_plant_raion', 'incineration_raion', 'oil_chemistry_raion', 'radiation_raion', 'railroad_terminal_raion', 'big_market_raion', 'nuclear_reactor_raion', 'detention_facility_raion', 'water_1line', 'big_road1_1line', 'railroad_1line'
]
data[cat_cols] = data[cat_cols].astype("category")
data = data.drop(columns=["0_6_all", "7_14_all", "id"])
data = data.reset_index(drop=True)
df = data

/tmp/ipykernel_608303/1061013581.py:29: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  data = data.filter(


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 28,321
Columns: 387
Use sampling: False (sample size: 28,321)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['ttk_km', 'basketball_km', 'hospice_morgue_km', 'power_transmission_line_km', 'sadovoe_km', 'big_road2_km', 'big_road1_km', 'oil_chemistry_km', 'catering_km', 'exhibition_km']
Rows remaining as candidates after top-10 filter: 21,379 (of 28,321)



#### Duplicate Report
Total duplicate rows: 9 (0.03% of dataset)
Duplicate rows ignoring target: 20 (0.07% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,product_type,sub_area,area_m,raion_popul,green_zone_part,indust_part,children_preschool,preschool_quota,preschool_education_centers_raion,children_school,school_quota,school_education_centers_raion,school_education_centers_top_20_raion,hospital_beds_raion,healthcare_centers_raion,university_top_20_raion,sport_objects_raion,additional_education_raion,culture_objects_top_25,culture_objects_top_25_raion,shopping_centers_raion,office_raion,thermal_power_plant_raion,incineration_raion,oil_chemistry_raion,radiation_raion,railroad_terminal_raion,big_market_raion,nuclear_reactor_raion,detention_facility_raion,full_all,male_f,female_f,young_all,young_male,young_female,work_all,work_male,work_female,ekder_all,ekder_male,ekder_female,0_6_male,0_6_female,7_14_male,7_14_female,0_17_all,0_17_male,0_17_female,16_29_all,16_29_male,16_29_female,0_13_all,0_13_male,0_13_female,raion_build_count_with_material_info,build_count_block,build_count_wood,build_count_frame,build_count_brick,build_count_monolith,build_count_panel,build_count_foam,build_count_slag,build_count_mix,raion_build_count_with_builddate_info,build_count_before_1920,build_count_1921-1945,build_count_1946-1970,build_count_1971-1995,build_count_after_1995,ID_metro,metro_min_avto,metro_km_avto,metro_min_walk,metro_km_walk,kindergarten_km,school_km,park_km,green_zone_km,industrial_km,water_treatment_km,cemetery_km,incineration_km,railroad_station_walk_km,railroad_station_walk_min,ID_railroad_station_walk,railroad_station_avto_km,railroad_station_avto_min,ID_railroad_station_avto,public_transport_station_km,public_transport_station_min_walk,water_km,water_1line,mkad_km,ttk_km,sadovoe_km,bulvar_ring_km,kremlin_km,big_road1_km,ID_big_road1,big_road1_1line,big_road2_km,ID_big_road2,railroad_km,railroad_1line,zd_vokzaly_avto_km,ID_railroad_terminal,bus_terminal_avto_km,ID_bus_terminal,oil_chemistry_km,nuclear_reactor_km,radiation_km,power_transmission_line_km,thermal_power_plant_km,ts_km,big_market_km,market_shop_km,fitness_km,swim_pool_km,ice_rink_km,stadium_km,basketball_km,hospice_morgue_km,detention_facility_km,public_healthcare_km,university_km,workplaces_km,shopping_centers_km,office_km,additional_education_km,preschool_km,big_church_km,church_synagogue_km,mosque_km,theater_km,museum_km,exhibition_km,catering_km,ecology,green_part_500,prom_part_500,office_count_500,office_sqm_500,trc_count_500,trc_sqm_500,cafe_count_500,cafe_sum_500_min_price_avg,cafe_sum_500_max_price_avg,cafe_avg_price_500,cafe_count_500_na_price,cafe_count_500_price_500,cafe_count_500_price_1000,cafe_count_500_price_1500,cafe_count_500_price_2500,cafe_count_500_price_4000,cafe_count_500_price_high,big_church_count_500,church_count_500,mosque_count_500,leisure_count_500,sport_count_500,market_count_500,green_part_1000,prom_part_1000,office_count_1000,office_sqm_1000,trc_count_1000,trc_sqm_1000,cafe_count_1000,cafe_sum_1000_min_price_avg,cafe_sum_1000_max_price_avg,cafe_avg_price_1000,cafe_count_1000_na_price,cafe_count_1000_price_500,cafe_count_1000_price_1000,cafe_count_1000_price_1500,cafe_count_1000_price_2500,cafe_count_1000_price_4000,cafe_count_1000_price_high,big_church_count_1000,church_count_1000,mosque_count_1000,leisure_count_1000,sport_count_1000,market_count_1000,green_part_1500,prom_part_1500,office_count_1500,office_sqm_1500,trc_count_1500,trc_sqm_1500,cafe_count_1500,cafe_sum_1500_min_price_avg,cafe_sum_1500_max_price_avg,cafe_avg_price_1500,cafe_count_1500_na_price,cafe_count_1500_price_500,cafe_count_1500_price_1000,cafe_count_1500_price_1500,cafe_count_1500_price_2500,cafe_count_1500_price_4000,cafe_count_1500_price_high,big_church_count_1500,church_count_1500,mosque_count_1500,leisure_count_1500,sport_count_1500,market_count_1500,green_part_2000,prom_part_2000,office_count_2000,office_sqm_2000,trc_count_2000,trc_sqm_2000,cafe_count_2000,cafe_sum_2000_min_price_avg,cafe_sum_2000_max_price_avg,cafe_avg_price_200

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,state,category,12701.0,44.85,5.0,"3.0, 2.0, 1.0, 4.0, 33.0"
1,material,category,8851.0,31.25,6.0,"1.0, 2.0, 5.0, 4.0, 6.0, 3.0"
2,ID_railroad_station_walk,category,24.0,0.08,132.0,"24.0, 47.0, 75.0, 39.0, 42.0, 4.0, 33.0, 2.0, 18.0, 28.0"
3,product_type,category,0.0,0.00,2.0,"Investment, OwnerOccupier"
4,sub_area,category,0.0,0.00,145.0,"Poselenie Sosenskoe, Nekrasovka, Poselenie Vnukovskoe, Poselenie Moskovskij, Poselenie Voskresenskoe, Mitino, Poselenie Filimonkovskoe, Krjukovo, Poselenie Shherbinka, Mar'ino"
5,culture_objects_top_25,category,0.0,0.00,2.0,"no, yes"
6,thermal_power_plant_raion,category,0.0,0.00,2.0,"no, yes"
7,incineration_raion,category,0.0,0.00,2.0,"no, yes"
8,oil_chemistry_raion,category,0.0,0.00,2.0,"no, yes"
9,radiation_raion,category,0.0,0.00,2.0,"no, yes"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
full_sq,28321.0,5.451831e+01,2.342759e+01,6.000000e+00,7.290000e+02
life_sq,21958.0,3.491880e+01,5.457491e+01,0.000000e+00,7.478000e+03
floor,28170.0,7.743876e+00,5.354901e+00,0.000000e+00,7.700000e+01
max_floor,19470.0,1.261002e+01,6.792349e+00,0.000000e+00,9.900000e+01
build_year,15468.0,3.171396e+03,1.612137e+05,0.000000e+00,2.005201e+07
num_room,19470.0,1.914176e+00,8.575572e-01,0.000000e+00,1.900000e+01
kitch_sq,19470.0,6.253364e+00,2.564831e+01,0.000000e+00,2.014000e+03
area_m,28321.0,1.860064e+07,2.145786e+07,2.081628e+06,2.060718e+08
raion_popul,28321.0,8.166703e+04,5.895374e+04,2.546000e+03,2.474690e+05
green_zone_part,28321.0,2.255320e-01,1.772803e-01,1.879375e-03,8.529228e-01


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                    rank                                       
ID_big_road1              1                           1   4430  15.64
                          2                           2   3455  12.20
                          3                          38   2721   9.61
                          4                          13   2704   9.55
                          5                           4   2169   7.66
ID_big_road2              1                           1   4136  14.60
                          2                           4   2376   8.39
                          3                          27   2040   7.20
                          4                          38   1204   4.25
                          5                          55   1167   4.12
ID_bus_terminal           1                           8   6966  24.60
                          2                           1   3787  13.37
                          3                           9   3328  11.75
                          4                           5   2604   9.19
                          5                           3   2085   7.36
ID_railroad_station_avto  1                          75   1702   6.01
                          2                         105   1653   5.84
                          3                          19   1583   5.59
                          4                           4   1308   4.62
                          5                          39   1301   4.59
ID_railroad_station_walk  1                        24.0   2677   9.45
                          2                        47.0   2196   7.75
                          3                        75.0   1702   6.01
                          4                        39.0   1301   4.59
                          5                        42.0   1221   4.31
ID_railroad_terminal      1                          32   8859  31.28
                          2                          50   5419  19.13
                          3                           5   5077  17.93
                          4                          83   3640  12.85
                          5                         121   1839   6.49
big_market_raion          1                          no  25547  90.21
                          2                         yes   2774   9.79
big_road1_1line           1                          no  27602  97.46
                          2                         yes    719   2.54
culture_objects_top_25    1                          no  27097  95.68
                          2                         yes   1224   4.32
detention_facility_raion  1                          no  26120  92.23
                          2                         yes   2201   7.77
ecology                   1                     no data   7813  27.59
                          2                        poor   7274  25.68
                          3                        good   6806  24.03
                          4                satisfactory   3364  11.88
                          5                   excellent   3064  10.82
incineration_raion        1                          no  25982  91.74
                          2                         yes   2339   8.26
material                  1                         1.0  13291  46.93
                          2                        <NA>   8851  31.25
                          3                         2.0   2714   9.58
                          4                         5.0   1386   4.89
                          5                         4.0   1305   4.61
nuclear_reactor_raion     1                          no  27550  97.28
                          2                         yes    771   2.72
oil_chemistry_raion       1                          no  28055  99.06
                          2                         yes    266   0.94
product_type              1                  Investment  17379  61.36
                          2               OwnerOccupier  109

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.944,-1.175,0.155,0.001,log,248443.5,2.724907e+17,exponential


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

n_steps = 5
now = df[date_col].max().normalize()

splits = {}

for step in range(n_steps):
    # Move split point back by 6 months each step
    test_time_max = now - pd.DateOffset(months=6 * step) + pd.DateOffset(day=1)
    test_time_min = now - pd.DateOffset(months=6 * (step + 1))

    if step == 0:
        test_mask = df[date_col] >= test_time_min
    else:
        test_mask = (df[date_col] >= test_time_min) & (df[date_col] < test_time_max)
    # Define indices
    test_idx = df.index[test_mask].to_numpy().tolist()
    train_idx = df.index[
        df[date_col] < test_time_min
    ].to_numpy().tolist()

    # Diagnostics
    print(f"\n=== Step {step} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].mean())
    print("Test target mean:", df.loc[test_idx, target_col].mean())

    splits[step] = {0: (train_idx, test_idx)}


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We always use 6 month of the data as test data and all prior data as training data. This simulate a model that is refit every half year. We create 5 splits going back 6 months each, starting from the newest date.",
    splits=splits,
    time_horizon=6,
    time_horizon_unit="months",
)


=== Step 0 ===
Train size: 25229 | Test size: 3092
Train target mean: 11.771233591123726
Test target mean: 11.832385125165297

=== Step 1 ===
Train size: 18847 | Test size: 4827
Train target mean: 11.752482748951802
Test target mean: 11.81531650654064

=== Step 2 ===
Train size: 12484 | Test size: 5306
Train target mean: 11.722513816733114
Test target mean: 11.814028162807226

=== Step 3 ===
Train size: 8104 | Test size: 3586
Train target mean: 11.700659268216672
Test target mean: 11.755326186683703

=== Step 4 ===
Train size: 5038 | Test size: 2571
Train target mean: 11.716871538118392
Test target mean: 11.663313899164537


## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to sberbank_housing_market_forecasting/019d7384-f3e1-7248-bbfc-f8115677fa9a


019d7384-f3e1-7248-bbfc-f8115677fa9a
ebac5616e017c7f27ac07ce36e573f84f499357d4be1af4e3367deaf14194726
